[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/02_tracking/B2_status_classification.ipynb)

# B2: Status Classification

---

## Learning Objectives

By the end of this notebook, you will be able to:
1. **Classify permit statuses** into meaningful categories
2. **Map raw status values** to standardized pipeline stages
3. **Handle status variations** across different data sources
4. **Create status hierarchies** for reporting

## Why This Matters

Government permit systems have dozens of status values:
- "Under Review", "In Review", "Review Pending"
- "Approved", "Conditional Approval", "Pending Final Action"
- "Incomplete", "Corrections Pending", "On Hold"

Users need simplified categories to understand the pipeline.

## Status Mapping Strategy

```
RAW STATUS                    →  CATEGORY
"In Review"                   →  UNDER_REVIEW
"Under Review"                →  UNDER_REVIEW
"Incomplete Pending Applicant"→  UNDER_REVIEW
"Approved"                    →  APPROVED
"Pending Final Action"        →  APPROVED
"Certificate of Occupancy"    →  COMPLETED
```

---

## NEW: Event-Based Status from permit_events

For projects with detailed Accela data, we can derive status from the **most recent event**:

```sql
-- Get current status from latest event
SELECT p.address_display, pe.stage, pe.action, pe.event_date
FROM permit_events pe
JOIN projects p ON pe.project_id = p.id
WHERE pe.event_date = (
    SELECT MAX(event_date) FROM permit_events 
    WHERE project_id = pe.project_id
)
```

**Stage values from Accela:**
- Completeness Review
- CEQA Determination  
- Staff Decision
- Appeal (Zoning Adjustments Board, City Council)
- Issuance
- Inspection

**Pro tip:** Check both `projects.status` (our classification) and `permit_events.stage` (Accela's tracking) for the most accurate picture.

---

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Find project root and setup environment
def find_project_root():
    """Find project root by looking for marker directories."""
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if (path / '00_config').exists() and (path / 'modules').exists():
            return path
    raise FileNotFoundError("Could not find project root")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

# Load config with resolved paths
import json
with open(ROOT / '00_config/berkeley_config.json') as f:
    CONFIG = json.load(f)

# Resolve relative paths to absolute
for key, value in CONFIG['paths'].items():
    if isinstance(value, str) and not value.startswith('http'):
        CONFIG['paths'][key] = str(ROOT / value)

print(f"✅ Project root: {ROOT}")
print(f"✅ Housing data: {CONFIG['paths']['housing_projects']}")

## 2. Load Projects

In [ ]:
# Load housing projects
housing_path = Path(CONFIG['paths']['housing_projects'])
df = load_csv(housing_path)

if df is not None:
    print(f"Loaded {len(df)} projects")
    
    # Show raw status values
    print(f"\nUnique status values:")
    for status in df['status'].unique():
        count = len(df[df['status'] == status])
        print(f"  {status}: {count}")

## 3. Classify All Projects

In [ ]:
# Apply classification
if df is not None:
    df['status_category'] = df['status'].apply(classify_project_status)
    
    # Summary
    summary = project_status_summary(df, 'status')
    print("Status Category Summary:")
    print("="*50)
    display(summary)
    
    # Total units by status
    if 'net_units' in df.columns:
        units_by_status = df.groupby('status_category')['net_units'].sum().sort_values(ascending=False)
        print(f"\nUnits by status category:")
        for status, units in units_by_status.items():
            print(f"  {status}: {units:,.0f} units")

## 4. Status Mapping Validation

In [ ]:
# Show mapping for each raw status
if df is not None:
    print("Status Mapping Validation:")
    print("="*60)
    
    mapping = df[['status', 'status_category']].drop_duplicates().sort_values('status_category')
    
    for _, row in mapping.iterrows():
        print(f"  '{row['status']}' -> {row['status_category']}")
    
    # Check for unknowns
    unknowns = df[df['status_category'] == 'unknown']
    if len(unknowns) > 0:
        print(f"\nWARNING: {len(unknowns)} projects with unknown status")
        print(unknowns['status'].value_counts())

## 5. Status Distribution by Year

In [ ]:
# Status by year
if df is not None and 'year' in df.columns:
    pivot = pd.pivot_table(
        df,
        index='year',
        columns='status_category',
        values='net_units',
        aggfunc='sum',
        fill_value=0
    )
    
    print("Units by Year and Status:")
    display(pivot)

## 6. Export Classified Data

In [ ]:
# Export with status categories
if df is not None:
    output_path = DATA_DIR / 'housing_projects_classified.csv'
    df.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

---

## Summary

This notebook:
- Classified raw status values into standard categories
- Validated status mappings
- Analyzed status distribution by year

**Next:** Run `B3_progress_indicators.ipynb` to track construction progress.